# Comparing motifs and P-values

Two topics come up again and again when working with motifs, though they are
rarely discussed in much detail: how to tell whether a match to a motif is any
good, and how to tell whether two motifs are alike. This guide covers both. The
first half explains motif scores and their P-values, which are what
[`scan_sequences()`][universalmotif.scan_sequences] uses to decide what counts
as a hit, and the second half covers comparing motifs with one another and
using those comparisons to merge similar motifs and to cluster them into trees.
For a quicker tour of the package as a whole, see the
[getting started](getting-started.ipynb) guide, and for every argument of every
function the [API reference](../reference/index.md).

In [ ]:
import warnings

import universalmotif as um

Most of the examples use the twelve example motifs that come with the package.
They are synthetic, each built around a well-known transcription factor binding
site, and come as count matrices with the family of the factor set. Let us
keep them in a dictionary as well, so that they can be picked out by name. We
will also need numpy and polars, and let polars show tables of up to 20 rows
in full:

In [ ]:
import numpy as np
import polars as pl

pl.Config.set_tbl_rows(20)
motifs = um.example_motifs()
by_name = {m.name: m for m in motifs}
um.summarize_motifs(motifs).select("name", "family", "consensus", "icscore")

## Scoring a match

A motif scores a sequence with its position weight matrix, or PWM, in which
each entry is the log2 ratio of the probability of a letter at that position
to its probability in the background. Let us look at the PWM of the AP-1
motif:

In [ ]:
ap1 = by_name["AP-1"]
um.convert_type(ap1, "PWM").matrix.round(2)

The score of a word as long as the motif is the sum of the PWM entries for its
letters, one from each column, and [`score_match()`][universalmotif.score_match]
does that sum for us. For example, here are the scores of the consensus site,
of the same site with its central C changed to a G, of a CRE site, and of a run
of As:

In [ ]:
words = ["ATGACTCAT", "ATGAGTCAT", "CTGACGTCA", "AAAAAAAAA"]
um.score_match(ap1, words)

The first two score the same, since the motif has no preference between C and
G at its centre, while the CRE site, which shares the TGAC half of the site but
not the rest, scores well below zero. The range of possible scores comes from
[`motif_range()`][universalmotif.motif_range], which adds up the lowest and the
highest entry of every column:

In [ ]:
um.motif_range(ap1)

Every score is computed with the PWM entries truncated to three decimals, the
same way scanning computes them, which is why the scores so far have at most
three decimal places. A raw score is hard to interpret on its own however. A score of 11.3 is
close to the best this motif can do, but a longer or more informative motif
would have a different range altogether. One way around this is to give
thresholds as fractions of the range of scores, which is what
[`motif_score()`][universalmotif.motif_score] does; here are the scores at 0%,
50%, 80% and 100% of the range:

In [ ]:
um.motif_score(ap1, [0, 0.5, 0.8, 1])

As we will see shortly though, a fraction of the range means rather different
things for different motifs. P-values are a much better yardstick.

## From scores to P-values

The P-value of a score is the probability that a random word, drawn one letter
at a time from the background, scores at least that much. The background stored
in the example motifs is uniform, so every 9-letter word has the same
probability, as [`prob_match()`][universalmotif.prob_match] shows:

In [ ]:
um.prob_match(ap1, ["ATGACTCAT", "AAAAAAAAA"])

This is simply 1/4<sup>9</sup>. For a motif this short we could therefore work
out a P-value by brute force. [`get_scores()`][universalmotif.get_scores] lists
the scores of all 4<sup>9</sup> = 262,144 possible words, and since they are
all equally likely, the P-value of a score is the fraction of them that reach
it. Let us do this for the score of the consensus site:

In [ ]:
all_scores = um.get_scores(ap1)
print(len(all_scores), (all_scores >= 11.319).sum(), (all_scores >= 11.319).mean())

Now let us ask [`motif_pvalue()`][universalmotif.motif_pvalue] for the same
P-value:

In [ ]:
um.motif_pvalue(ap1, score=11.319)

The two agree exactly. Listing every word quickly becomes impossible however,
since a motif of 20 positions already has about 10<sup>12</sup> of them, and so
[`motif_pvalue()`][universalmotif.motif_pvalue] works differently. It builds the
full distribution of scores one column at a time, adding each column's scores
and their background probabilities to the distribution of the columns before
it. Because scores are kept on a grid of steps of 0.001, the distribution stays
small, and the result is exact for scores on that grid whatever the length of
the motif or the background. Several scores can be given at once:

In [ ]:
um.motif_pvalue(ap1, score=[0, 5, 10, 13.356])

Note that even the maximum score has a P-value above 1/4<sup>9</sup>, because
two words reach it, one with a C at the centre and one with a G. Scores outside
the range of the motif are fine too, and get P-values of one and zero, with a
warning about the score that no word can reach:

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    p = um.motif_pvalue(ap1, score=[-100, 100])
print(p)
print(caught[0].message)

## From P-values to score thresholds

More often we want to go the other way, and find the score threshold for a
P-value cutoff. Given `pvalue` instead of `score`,
[`motif_pvalue()`][universalmotif.motif_pvalue] returns the lowest score on its
grid whose P-value is no greater than the cutoff:

In [ ]:
cutoffs = [1e-2, 1e-3, 1e-4, 1e-5, 1e-6]
thresholds = um.motif_pvalue(ap1, pvalue=cutoffs)
thresholds

Let us check the threshold for 1e-4 by listing every word that reaches it with
[`get_matches()`][universalmotif.get_matches], and adding up their
probabilities:

In [ ]:
matches = um.get_matches(ap1, thresholds[2])
print(len(matches), matches[:6])
um.prob_match(ap1, matches).sum()

These 26 words have a total probability just under 1e-4, as they should. The
last two thresholds deserve a closer look. The threshold for 1e-5 is 13.04,
which no word actually scores. Here are the four highest scores the motif can
give:

In [ ]:
np.unique(all_scores)[-4:]

Any threshold above 13.039 and up to 13.356 selects the same two words, those
with the maximum score, and 13.04 is the lowest such threshold on the grid. In
other words, a threshold need not be a score that any word can reach. The
threshold for 1e-6 is infinite, because even the best possible match, with its
P-value of about 7.6e-6, is not that unlikely. No word reaches an infinite
threshold, so [`get_matches()`][universalmotif.get_matches] returns nothing and
scanning with such a cutoff finds no hits at all, which a warning points out
along with the smallest P-value the motif can give:

In [ ]:
print(um.get_matches(ap1, thresholds[-1]))
targets = um.example_sequences()
with warnings.catch_warnings(record=True) as caught:
    n = um.scan_sequences(ap1, targets, pvalue=1e-6).height
print(n)
print(caught[0].message)

This is worth keeping in mind with short or weak motifs. A cutoff as strict as
1e-6 is perfectly reasonable for a long motif, but most of the example motifs
simply cannot reach it:

In [ ]:
um.motif_pvalue(motifs, pvalue=1e-6)

With a list of motifs, [`motif_pvalue()`][universalmotif.motif_pvalue] gives
one value per motif. Only the TATA-box, ETS and CArG-box motifs, the three
10-position motifs, can produce a match with a P-value of 1e-6 or less.

## Choosing a threshold

Let us now come back to thresholds given as fractions of the range of scores.
For each example motif, here is the score at 80% of its range, the P-value of
that score, and for comparison the threshold for a P-value of 1e-4:

In [ ]:
rows = []
for m in motifs:
    score80 = um.motif_score(m, 0.8)
    rows.append({
        "name": m.name,
        "width": m.width,
        "score_80pct": score80,
        "pvalue_80pct": um.motif_pvalue(m, score=score80),
        "threshold_1e-4": um.motif_pvalue(m, pvalue=1e-4),
    })
pl.DataFrame(rows)

The same 80% threshold gives P-values ranging from about 0.0013 for the GC-box
to about 0.015 for GATA, a more than ten-fold difference, which means that
scanning with it would be far more lenient with some motifs than with others. A
P-value cutoff on the other hand is equally strict for every motif, even though
the scores it corresponds to vary.

Which P-value to choose is another matter. A useful way to think about it is
the number of hits expected by chance: a 9-position motif has 992 windows in a
sequence of 1000 bases, on each of two strands, so scanning the 100 example
sequences at a cutoff of 1e-4 should turn up about 20 chance hits. The example
sequences come as targets, with a W-box site planted in 60 of them, and
controls with nothing planted. Let us scan both at a few cutoffs and count the
sequences with at least one hit:

In [ ]:
wbox = by_name["W-box"]
controls = um.example_sequences("controls")

def count_hits(motif, cutoffs):
    rows = []
    for p in cutoffs:
        t = um.scan_sequences(motif, targets, pvalue=p)
        c = um.scan_sequences(motif, controls, pvalue=p)
        rows.append({
            "pvalue": p,
            "target_hits": t.height,
            "target_seqs": t["sequence"].n_unique(),
            "control_hits": c.height,
            "control_seqs": c["sequence"].n_unique(),
        })
    return pl.DataFrame(rows)

count_hits(wbox, [1e-3, 1e-4, 1e-5])

At 1e-3, most sequences in both sets have a hit, and the planted sites are lost
among the chance matches. At 1e-4 the targets clearly stand out, though only 30
target sequences have a hit, so at least half of the 60 planted sites are
missed; sites sampled from a motif are rarely perfect matches to it. At 1e-5
almost nothing is left in either set. So there is a trade-off between finding the real sites and keeping out the chance ones,
and it is important to remember that there is likely never one single cutoff
that works in every situation. It depends on the motif, on how much sequence
is scanned, and on how many false positives can be tolerated.

## Backgrounds

There is something odd about the controls in the table above: at 1e-4 they
have 10 hits, only half of the roughly 20 expected by chance. The reason is the
background. The background of a motif is used twice, once for the log ratios in
its PWM and once for the probability of each word, and the example motifs have
a uniform background while the example sequences are AT-rich, much like plant
promoters. [`get_bkg()`][universalmotif.get_bkg] shows by how much:

In [ ]:
bkg = um.get_bkg(targets, k=1, rc=True)
at_rich = dict(zip(bkg["kmer"], bkg["probability"]))
bkg

The W-box is fairly rich in G and C, which are uncommon in these sequences, so
its high-scoring words turn up less often than a uniform background predicts,
and its P-values are conservative for these sequences. The `bkg_probs`
argument of [`motif_pvalue()`][universalmotif.motif_pvalue] computes a P-value
or threshold as if the motif had a different background, which changes both
the PWM and the word probabilities. Setting the background of the motif itself
with `replace()` before it is turned into a PWM has the same effect:

In [ ]:
wbox_at = wbox.replace(bkg=at_rich)
print(um.motif_pvalue(wbox, pvalue=1e-4))
print(um.motif_pvalue(wbox, pvalue=1e-4, bkg_probs=at_rich))
print(um.motif_pvalue(wbox_at, pvalue=1e-4))

Scanning uses the background stored in the motif, so let us scan again with
the AT-rich version:

In [ ]:
count_hits(wbox_at, [1e-4, 1e-5])

Now the controls have 21 hits at 1e-4, very close to the 20 or so expected, so
the P-values are much better calibrated for these sequences. At the same
cutoffs more target sequences have a hit as well, 46 rather than 30 at 1e-4
and 12 rather than 1 at 1e-5. One last subtlety is that the background is only
built into the scores when a motif is converted to a PWM.
Changing the background of a motif that is already a PWM leaves its scores as
they are, and changes only the probabilities of the words:

In [ ]:
pwm = um.convert_type(ap1, "PWM")
pwm_at = pwm.replace(bkg=at_rich)
print(um.score_match(pwm, "ATGACTCAT"), um.score_match(pwm_at, "ATGACTCAT"))
print(um.prob_match(pwm, "ATGACTCAT"), um.prob_match(pwm_at, "ATGACTCAT"))

So to move a motif to a new background, it is best to set the background on its
counts or probabilities and convert to a PWM from there.
The [Choosing a background](background-models.ipynb) guide goes into how to
pick one, with shuffled and composition-matched controls among the options.

## Comparing columns

Let us now turn to comparing motifs. Motifs are compared one aligned column at
a time, and [`compare_motifs()`][universalmotif.compare_motifs] uses the
Pearson correlation of the probabilities in each pair of columns. Two columns
can be compared directly with
[`compare_columns()`][universalmotif.compare_columns]:

In [ ]:
c1 = [0.7, 0.1, 0.1, 0.1]
c2 = [0.5, 0.0, 0.2, 0.3]
um.compare_columns(c1, c2, "PCC"), np.corrcoef(c1, c2)[0, 1]

[`compare_columns()`][universalmotif.compare_columns] knows a number of other
metrics too, which can be handy for getting a feel for how they behave. Some
are distances, where lower means more alike, and others are similarities:

In [ ]:
metrics = ["PCC", "WPCC", "EUCL", "WEUCL", "SEUCL", "MAN",
           "HELL", "SW", "BHAT", "KL", "ALLR", "ALLR_LL"]
{m: round(um.compare_columns(c1, c2, m), 4) for m in metrics}

Note that KL and ALLR take logarithms of the probabilities, so the zero in the
second column, facing a nonzero probability in the first, gives them an
infinite score. Zeros in both columns at the same letter add nothing, but in
general these two metrics need columns without zeros, which a small
pseudocount takes care of:

In [ ]:
c2_pseudo = [(x + 0.01) / 1.04 for x in c2]
um.compare_columns(c1, c2_pseudo, "KL"), um.compare_columns(c1, c2_pseudo, "ALLR")

## Comparing motifs

To compare two whole motifs, [`compare_motifs()`][universalmotif.compare_motifs]
slides one along the other, and for DNA and RNA motifs along its reverse
complement too. At each offset it adds up the correlations of the overlapping
columns, and it keeps the alignment with the highest sum among those that
overlap by at least `min_overlap` columns. To see how this works, let us make
a few small motifs: one site, a shorter piece of it, the reverse complement of
a shifted version of it, and something unrelated.

In [ ]:
site = um.create_motif("ATTGACCGAT", name="site", pseudocount=1, nsites=20)
part = um.create_motif("TTGACCG", name="part")
flipped = um.motif_rc(um.create_motif("GCTTGACCGA")).replace(name="flipped")
other = um.create_motif("CCCCCGGGCG", name="other")
small = [site, part, flipped, other]
print(flipped.consensus)
um.compare_motifs(small).round(3)

By default the result is a matrix of the mean column correlation of the best
alignment of each pair, with the motifs as both rows and columns. `part`
matches `site` perfectly, whereas `flipped` only matches it once reverse
complemented, and even then its best score is 0.852 rather than 1. The reason
is that a shifted alignment overlaps by nine columns, eight of them identical
and one pairing an A with a C, and all overlapping columns count towards the
score. Without trying reverse complements, `flipped` looks rather unlike the
site, as the first row of the matrix shows:

In [ ]:
um.compare_motifs(small, rc=False)[0].round(3)

To see the alignments themselves, give `compare_to`, one or more motifs to use
as queries, either as 0-based indices or as names. The result is then a
DataFrame with one row per pair, ordered by significance. Only pairs with a
q-value of at most `qvalue` are kept, so let us set it to one to see them all:

In [ ]:
um.compare_motifs(small, compare_to="site", qvalue=1)

The `offset` is where the target starts relative to the query, `strand` is `-`
when the query was reverse complemented to line them up, and `overlap` is the
number of aligned columns. The two consensus columns show the aligned parts of
each motif, each in its own orientation, so on the `-` strand the reverse
complement of one lines up with the other: the reverse complement of TCGGTCAAG
is CTTGACCGA, which differs from ATTGACCGA only at its first position. The
`score` is the mean column correlation, the same as in the matrix.

`min_overlap` sets how short an alignment may be. Shorter overlaps let motifs
match by their edges, which can be what we want for motifs of quite different
lengths, but also makes it easier for unrelated motifs to find a few columns
that agree. For example, AP-1 and the CArG-box are aligned by only five columns
with the default of 5, so let us compare that with an overlap of at least 9:

In [ ]:
carg = by_name["CArG-box"]
cols = ["target", "offset", "strand", "overlap", "score",
        "subject_consensus", "target_consensus"]
for overlap in (5, 9):
    hits = um.compare_motifs([ap1, carg], compare_to=0, qvalue=1, min_overlap=overlap)
    print(hits.select(cols))

Requiring nine columns forces the whole of AP-1 to be aligned, and the score
drops to around zero. A `min_overlap` longer than the shorter motif is the same
as asking for the shorter motif to be aligned in full.

Now let us compare all of the example motifs. The matrix is easier to read
with the names alongside it:

In [ ]:
names = [m.name for m in motifs]
scores = um.compare_motifs(motifs)
pl.DataFrame(scores.round(2), schema=names).insert_column(0, pl.Series("motif", names))

Note that the scores are symmetric, since swapping the query and target gives
the same best alignment. The most similar pairs are AP-1 and the W-box at 0.81,
followed by the Telo-box and the CCAAT-box at 0.75 and the Telo-box and the
CArG-box at 0.73.

## How significant is a comparison?

A mean correlation of 0.81 sounds high, but whether it is surprising depends on
how long the alignment is and on what columns in general look like. So
[`compare_motifs()`][universalmotif.compare_motifs] also gives each alignment a
P-value: the probability of a sum of column correlations at least as high, for
an alignment of the same length, under a null model of random columns. This is
computed exactly from the distribution of column correlations, then
Bonferroni-corrected for the number of alignments that were tried. The
q-values are Benjamini-Hochberg adjusted P-values within each query. With the
default `qvalue` of 0.1, comparing AP-1 against everything else keeps just two
rows:

In [ ]:
um.compare_motifs(motifs, compare_to="AP-1")

Apart from AP-1 itself, only the W-box passes. The two motifs share a TGAC,
which [`view_motifs()`][universalmotif.view_motifs] makes easy to see, since it
aligns the motifs the same way before drawing them:

In [ ]:
fig = um.view_motifs([ap1, wbox])

The P-values and q-values are also available as matrices, with `matrix_out`
set to `"pvalue"` or `"qvalue"`. Unlike the scores these need not be
symmetric, since both the null distribution and the adjustment are worked out
for each query. Let us look at the
q-values of the first six motifs:

In [ ]:
qvalues = um.compare_motifs(motifs, matrix_out="qvalue")
first = names[:6]
pl.DataFrame(qvalues[:6, :6].round(4), schema=first).insert_column(0, pl.Series("motif", first))

There are two null models to choose from. The default `"empirical"` null uses
the columns of all the motifs given, which describes what columns actually look
like but also means that the P-values depend on which motifs are compared.
With only AP-1 and the W-box, for example, the P-value of their alignment is
about twice as large:

In [ ]:
um.compare_motifs([ap1, wbox], compare_to="AP-1", qvalue=1)

The `"parametric"` null, which is for DNA and RNA motifs only, instead describes
random columns with a Dirichlet distribution around the background, given by
`bkg` and uniform by default. Its P-values do not depend on the other motifs,
which suits small sets of motifs where the empirical null has few columns to
go on:

In [ ]:
cols = ["target", "score", "pvalue", "qvalue"]
print(um.compare_motifs(motifs, compare_to="AP-1", null="parametric").select(cols))
print(um.compare_motifs([ap1, wbox], compare_to="AP-1", null="parametric").select(cols))

The parametric P-value for AP-1 against the W-box is the same in both sets,
and the q-value changes only because there are fewer comparisons to adjust
for. Note that a perfect match of a motif to itself is so unlikely under the
parametric null that its P-value comes out as exactly zero.

## Merging motifs

Motifs found in different experiments, or by different tools, are often
variants of one another, and it can be useful to combine them into a single
consensus motif. Let us make three variants of a W-box: two from slightly
different sets of sites, one of them shifted by a position, and a third which
is reverse complemented.

In [ ]:
v1 = um.create_motif(
    ["CTTTGACCAA", "GTTTGACCAA", "CTTTGACTAA", "CTTTGACCAT"], name="W-box v1"
)
v2 = um.create_motif(["TTTGACCAAT", "TTTGACTAAT", "TTTGACCAAA"], name="W-box v2")
v3 = um.motif_rc(um.create_motif(
    ["ACTTTGACCA", "ACTTTGACCA", "AGTTTGACTA", "ACTTTGACCA", "TCTTTGACCA"]
)).replace(name="W-box v3")
variants = [v1, v2, v3]
[v.consensus for v in variants]

[`merge_motifs()`][universalmotif.merge_motifs] takes the motif with the highest
total information content as an anchor, aligns the others to it as
[`compare_motifs()`][universalmotif.compare_motifs] would, reverse complementing
them where that fits better, and averages the aligned probabilities. The merged
motif spans every column that any of the motifs covers:

In [ ]:
merged = um.merge_motifs(variants, new_name="W-box merged")
print(merged)

In [ ]:
fig = um.view_motifs(variants + [merged])

By default every motif counts equally. With `weighted=True` the probabilities
are instead weighted by the number of sites behind each motif, which gives the
five sites of `W-box v3` more say than the three of `W-box v2`. The differences
are small here, since the variants agree on most positions:

In [ ]:
um.merge_motifs(variants, weighted=True).matrix.round(2)

It is important to note that [`merge_motifs()`][universalmotif.merge_motifs]
merges whatever it is given. It does not check whether the motifs are actually
alike, and an unrelated motif is merged in just the same, as long as it can
overlap the anchor by `min_overlap` columns. Adding the GC-box changes one
position of the consensus:

In [ ]:
with_gc = um.merge_motifs(variants + [by_name["GC-box"]])
print(with_gc.name, with_gc.consensus)

Deciding which motifs to merge is the job of
[`merge_similar()`][universalmotif.merge_similar]. It compares all the motifs,
links every pair with a q-value of at most `qvalue` in either direction, and
merges each connected group of motifs. Let us mix the variants in with the
example motifs, and first look at the clusters only:

In [ ]:
mixed = motifs + variants
clusters = um.merge_similar(mixed, return_clusters=True)
clusters.select("name", "consensus", "motif_i", "cluster")

The three variants form one cluster, and every other motif is on its own,
including the example W-box, whose sites are rather more varied. Without
`return_clusters` we get back one motif per cluster, with the variants
merged:

In [ ]:
[m.name for m in um.merge_similar(mixed)]

The default cutoff of 0.01 is fairly strict. Loosening it pulls more motifs
together, and at 0.1 one cluster gathers six of them:

In [ ]:
loose = um.merge_similar(mixed, qvalue=0.1, return_clusters=True)
loose.filter(pl.col("cluster") == 1).select("name", "consensus", "cluster")

To see why, we can list the pairs that pass the same cutoff with
[`compare_motifs()`][universalmotif.compare_motifs], leaving out the
comparisons of each motif with itself:

In [ ]:
links = um.compare_motifs(mixed, compare_to=list(range(len(mixed))), qvalue=0.1)
links.filter(pl.col("subject") != pl.col("target")).select(
    "subject", "target", "overlap", "score", "qvalue"
)

The variants are linked to the example W-box, which is linked to AP-1 through
their shared TGAC, and `W-box v2` is also linked to the CArG-box by a
five-column overlap of their AT-rich ends. Since clusters are connected groups,
a single link is enough to join two motifs to the same cluster, however unlike
they are otherwise. Whether that is desirable depends on what the motifs are
for, so it is always worth looking at the clusters before trusting the merged
motifs.

## Motif trees

A tree is often the clearest way of seeing how a set of motifs relate to one
another. [`motif_tree()`][universalmotif.motif_tree] compares the motifs with
[`compare_motifs()`][universalmotif.compare_motifs], turns the mean column
correlations into distances as (1 - score) / 2, so that identical motifs are
at a distance of zero and perfectly anticorrelated ones at one, and clusters
the distances hierarchically. The default is complete linkage, and `method`
offers the others:

In [ ]:
tree = um.motif_tree(motifs)
tree

Note that `min_overlap` is 6 here by default, rather than the 5 of
[`compare_motifs()`][universalmotif.compare_motifs]. The returned
[`MotifTree`][universalmotif.MotifTree] holds the clustering in a few forms. The
tip labels come in input order, `order` gives the motifs in the order they
appear down the tree, and `height` the height of each merge:

In [ ]:
print([tree.labels[i] for i in tree.order])
tree.height.round(3)

The tree is also available as a Newick string, ready for any tree viewer, and
[`MotifTree.write_newick()`][universalmotif.MotifTree.write_newick] saves it to
a file:

In [ ]:
print(tree.newick)

import tempfile, pathlib
newick_path = pathlib.Path(tempfile.mkdtemp()) / "motifs.nwk"
tree.write_newick(newick_path)

For further analysis, `linkage` gives the same tree as a SciPy linkage matrix,
so all of the `scipy.cluster.hierarchy` functions can be used with it, such
as `fcluster()` to cut it into a set number of clusters:

In [ ]:
from scipy.cluster import hierarchy

hierarchy.fcluster(tree.linkage, 4, criterion="maxclust")

Cutting a tree is also built in, as
[`MotifTree.cutree()`][universalmotif.MotifTree.cutree], which takes either a
number of groups `k` or a height `h` to cut at. Groups are numbered from zero
in the order they first appear among the motifs:

In [ ]:
groups = tree.cutree(k=4)
pl.DataFrame({"motif": names, "group": groups})

The four groups are the same as those from `fcluster()`, only numbered
differently. Cutting lower down, at a height of 0.3, splits them further:

In [ ]:
tree.cutree(h=0.3)

[`MotifTree.plot()`][universalmotif.MotifTree.plot] draws the tree with
matplotlib. With `tip_logos=True` each motif's logo is drawn at its tip,
aligned to the others as in [`view_motifs()`][universalmotif.view_motifs], which
makes it much easier to see what drives the clustering. Branches can be
coloured by group, given either as one value per motif or as a motif
attribute, and a branch takes the colour of a group when every motif below it
belongs to that group. Let us colour the branches by the four groups
from [`MotifTree.cutree()`][universalmotif.MotifTree.cutree], leaving out the
legend since the groups have no names:

In [ ]:
ax = tree.plot(groups=groups, tip_logos=True, legend=False)

Tip logos need the rectangular or slanted layouts. The circular and fan
layouts suit larger trees, and `tip_size` can size the tips by a motif
attribute such as the total information content. Let us colour the branches by
family this time. The colours are taken in turn from the matplotlib colour
cycle, in the order the families first appear, so the TBP family of the
TATA-box is blue and the bZIP family orange; `colours` sets them explicitly
instead. The legends beside the tree name the families and show the tip sizes.

In [ ]:
ax = tree.plot(layout="circular", groups="family", tip_size="icscore")

The three bZIP motifs do not all end up together. CRE and the G-box, which
share an ACGT core, sit next to each other, but AP-1 is closest to the W-box,
as we saw earlier.

Finally, [`motif_tree()`][universalmotif.motif_tree] also accepts a square
matrix of distances instead of motifs, for trees built from some other measure
of similarity. In that case the labels have to be given, and since there are
no motifs, there can be no tip logos. Building the same distances as before by
hand gives the same tree:

In [ ]:
distances = (1 - um.compare_motifs(motifs, min_overlap=6)) / 2
same_tree = um.motif_tree(distances, labels=names)
same_tree.newick == tree.newick

## Where next

The [API reference](../reference/index.md) describes every argument of the
functions used here, in its [P-values and scores](../reference/scoring.md) and
[comparison and clustering](../reference/comparison.md) pages. The P-values of
individual matches are what scanning and enrichment testing are built on, and
the [scanning](../reference/scanning.md) and
[enrichment](../reference/enrichment.md) pages cover those.